In [ ]:
import Pkg
Pkg.add("ITensors")
Pkg.add("DataFrames")
Pkg.instantiate()

In [ ]:
using DelimitedFiles
using Pkg
using XLSX
using DataFrames
Pkg.activate("../")  # Activate the main project
using Dleto
using Random
using ITensors
using Statistics
using Plots



In [ ]:
function process_trade_agreement_hypergraph(k::Int; seed::Union{Int,Nothing}=nothing)

    # Load the Trade Agreement Excel file
    xlsx_file = "Trade-Agreements/AllRTAs.xlsx"
    sheet = XLSX.readtable(xlsx_file, 1)
    df = DataFrame(sheet)

    # Filter rows where Status is "In Force"
    in_force_df = filter(row -> row[:Status] == "In Force", df)

    # Extract all nations from column "CurrentSignatories" (semicolon-separated)
    all_nations = String[]
    for row in eachrow(in_force_df)
        nations = split(strip(row[:CurrentSignatories]), ';')
        append!(all_nations, strip.(nations))
    end
    unique_nations = sort(unique(all_nations))
    println("Total unique nations in dataset: ", length(unique_nations))
    println("Total agreements (simplices) in dataset: ", size(in_force_df, 1))
    # Choose random k nations
    if seed !== nothing
        Random.seed!(seed)
        println("Random seed set to $seed")
    end
    random_k_nations = Set(shuffle(unique_nations)[1:k])
    println("Filtering to random $k nations: $(sort(collect(random_k_nations)))")

    # Filter simplices to only those with all nodes in random_k_nations
    simplices = Vector{Vector{String}}()
    for row in eachrow(in_force_df)
        nations = split(strip(row[:CurrentSignatories]), ';')
        nations = strip.(nations)
        if all(n -> n in random_k_nations, nations)
            push!(simplices, nations)
        end
    end
    if length(simplices) == 0
        println("No agreements found for this random subset. Try increasing k or changing the seed.")
        return nothing
    end
    # Check for at least one simplex with at least three nodes
    if all(length(simplex) < 3 for simplex in simplices)
        println("No simplex with at least three nodes found. Try increasing k or changing the seed.")
        return nothing
    end

    
    println("Total number of simplices: ", length(simplices))
    println("Simplex size distribution:")
    size_counts = Dict{Int, Int}()
    for simplex in simplices
        sz = length(simplex)
        size_counts[sz] = get(size_counts, sz, 0) + 1
    end
    for sz in sort(collect(keys(size_counts)))
        println("  Size $sz: $(size_counts[sz]) simplices")
    end

    println("First few simplices:")
    for i in 1:min(10, length(simplices))
        println("  Simplex $i: $(simplices[i])")
    end

    # Map each nation to an integer index (only for the random k)
    unique_k_nations = sort(collect(random_k_nations))
    nation_to_idx = Dict(nation => idx for (idx, nation) in enumerate(unique_k_nations))

    # Convert simplices to integer indices
    simplices_idx = [map(n -> nation_to_idx[n], simplex) for simplex in simplices]

    # Create temporal_simplices (no timestamps, just node vectors)
    temporal_simplices = [(simplices_idx[i], 0, length(simplices_idx[i])) for i in 1:length(simplices_idx)]
    println("Created temporal hypergraph with ", length(temporal_simplices), " temporal simplices")
    println("Number of nodes: ", length(unique_k_nations))
    println("\nReady for tensor creation!")

    # 3-mode tensor creation
    n_nodes = length(unique_k_nations)

    function get_3node_combinations(nodes)
        if length(nodes) < 3
            return []
        elseif length(nodes) == 3
            return [nodes]
        else
            combinations = []
            for i in 1:length(nodes)-2
                for j in i+1:length(nodes)-1
                    for k in j+1:length(nodes)
                        push!(combinations, [nodes[i], nodes[j], nodes[k]])
                    end
                end
            end
            return combinations
        end
    end

    i = Index(n_nodes, "node1")
    j = Index(n_nodes, "node2")
    k = Index(n_nodes, "node3")
    Triangles = ITensor(Float64, i, j, k)

    processed_simplices = 0
    total_triangles = 0
    skipped_small = 0
    for ts in temporal_simplices
        triangle_combinations = get_3node_combinations(ts[1])
        if isempty(triangle_combinations)
            skipped_small += 1
            continue
        end
        for triangle in triangle_combinations
            node_indices = triangle
            if length(node_indices) == 3
                a, b, c = node_indices[1], node_indices[2], node_indices[3]
                Triangles[i=>a, j=>b, k=>c] += 1.0
                Triangles[i=>a, j=>c, k=>b] += 1.0
                Triangles[i=>b, j=>a, k=>c] += 1.0
                Triangles[i=>b, j=>c, k=>a] += 1.0
                Triangles[i=>c, j=>a, k=>b] += 1.0
                Triangles[i=>c, j=>b, k=>a] += 1.0
                total_triangles += 1
            end
        end
        processed_simplices += 1
    end

    T_array = Array(Triangles, i, j, k)
    non_zero_entries = count(x -> x != 0, T_array)
    total_entries = length(T_array)
    println("✅ 3-mode triangle tensor created!")
    println("Processed simplices: $processed_simplices")
    println("Skipped simplices (< 3 nodes): $skipped_small")
    println("Total triangles extracted: $total_triangles")
    println("Non-zero tensor entries: $non_zero_entries / $total_entries")
    println("Sparsity: $(round((total_entries - non_zero_entries) / total_entries * 100, digits=1))%")
    if non_zero_entries > 0
        println("Max triangle count: $(round(maximum(T_array[T_array .> 0]), digits=2))")
        println("Mean triangle count: $(round(mean(T_array[T_array .> 0]), digits=2))")
    else
        println("Max triangle count: N/A (no triangles)")
        println("Mean triangle count: N/A (no triangles)")
    end

    # Subsample tensor is the same as the full tensor in this filtered case (k x k x k)
    max = n_nodes
    i_small = Index(max, "node1_small")
    j_small = Index(max, "node2_small")
    k_small = Index(max, "node3_small")
    SmallTrade = ITensor(i_small, j_small, k_small)
    T_small_array = T_array[1:max, 1:max, 1:max]
    for a in 1:max, b in 1:max, c in 1:max
        if T_small_array[a, b, c] != 0
            SmallTrade[i_small=>a, j_small=>b, k_small=>c] = T_small_array[a, b, c]
        end
    end
    small_non_zero = count(x -> x != 0, T_small_array)
    small_total = length(T_small_array)
    println("✅ Subsample tensor created!")

    # Check for empty, all-zero, NaN, or Inf values before stratify
    arr = Array(SmallTrade, inds(SmallTrade)...)
    if all(x -> x == 0, arr)
        println("Tensor is empty or all zeros. Skipping stratify.")
        return nothing
    elseif any(isnan, arr) || any(isinf, arr)
        println("Tensor contains NaN or Inf values. Cannot stratify.")
        return nothing
    else
        println("\n🔬 Performing stratified decomposition on SmallTrade tensor...")
        @time SmallTrade_strat, Xs_strat_small = stratify(SmallTrade)
        println("✅ SmallTrade stratification complete!")
    end

    # Plot SmallTrade and SmallTrade_strat side by side
    p1 = plot_tensor(SmallTrade; title="Original SmallTrade", color=:blue)
    p2 = plot_tensor(SmallTrade_strat; title="Stratified SmallTrade", color=:red)
    Plots.plot(p1, p2; layout=(1,2), size=(800,400))

    return (
        Triangles=Triangles,
        SmallTrade=SmallTrade,
        SmallTrade_strat=SmallTrade_strat,
        Xs_strat_small=Xs_strat_small,
        temporal_simplices=temporal_simplices
    )
end

# Example usage:
# result = process_trade_agreement_hypergraph(50)

In [ ]:
result0 = process_trade_agreement_hypergraph(30)

In [ ]:
result1 = process_trade_agreement_hypergraph(30)

In [ ]:
result2 = process_trade_agreement_hypergraph(30)

In [ ]:
result3 = process_trade_agreement_hypergraph(50)

In [ ]:
result4 = process_trade_agreement_hypergraph(50)

In [ ]:
result5 = process_trade_agreement_hypergraph(50)

In [ ]:
result6 = process_trade_agreement_hypergraph(80)

In [ ]:
result7 = process_trade_agreement_hypergraph(80)

In [ ]:
result8 = process_trade_agreement_hypergraph(80)

In [ ]:
result9 = process_trade_agreement_hypergraph(100)

In [ ]:
result10 = process_trade_agreement_hypergraph(100)

In [ ]:
result11 = process_trade_agreement_hypergraph(100)

In [ ]:
result12 = process_trade_agreement_hypergraph(150)

In [ ]:
result13 = process_trade_agreement_hypergraph(150)

In [ ]:
result14 = process_trade_agreement_hypergraph(150)